# Claude + Unsiloed: Document Processing with Tool Use

This notebook shows how to give Claude structured access to Unsiloed's document processing API using [Anthropic's tool-use feature](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview).

**What we'll build:**
1. Define Unsiloed API endpoints as tool-use schemas
2. Build a tool executor that dispatches Claude's tool calls to the Unsiloed API
3. Run an agentic loop where Claude autonomously extracts data, classifies documents, and parses content

**Prerequisites:**
- An [Unsiloed API key](https://www.unsiloed.ai)
- An [Anthropic API key](https://console.anthropic.com)

## 1. Setup

Install dependencies and load API keys from your `.env` file.

In [ ]:
%pip install anthropic requests python-dotenv -q

In [ ]:
import os
import json
import time
import requests
import anthropic
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
UNSILOED_API_KEY = os.getenv("UNSILOED_API_KEY")
UNSILOED_BASE_URL = "https://prod.visionapi.unsiloed.ai"
UNSILOED_HEADERS = {"api-key": UNSILOED_API_KEY}

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Setup complete.")

## 2. Define Unsiloed Tools

We define 5 tools that map to Unsiloed's core API endpoints. Each tool follows [Anthropic's tool format](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview) with a `name`, `description`, and `input_schema`.

| Tool | Endpoint | Purpose |
|------|----------|---------|
| `unsiloed_parse_document` | `POST /parse` | Parse documents into structured chunks |
| `unsiloed_extract_data` | `POST /v2/extract` | Extract structured data using a JSON schema |
| `unsiloed_classify_document` | `POST /classify` | Classify a PDF into categories |
| `unsiloed_split_document` | `POST /splitter` | Split a multi-doc PDF by category |
| `unsiloed_get_job_result` | `GET /{type}/{job_id}` | Poll for async job results |

All Unsiloed operations are **asynchronous** — they return a `job_id` immediately, and you poll with `unsiloed_get_job_result` to get the results.

> **Tip:** You can also copy these schemas from the [Claude Integration docs page](https://docs.unsiloed.ai/document-processing/claude-integration).

In [ ]:
tools = [
    {
        "name": "unsiloed_parse_document",
        "description": "Parse a document into structured chunks with element detection, text extraction, and reading order analysis. Use this tool when the user wants to break a document into its structural components (text blocks, tables, images, headers, footers) or convert a document to markdown. You must provide a publicly accessible URL to the document. Supported formats include PDF, PNG, JPEG, TIFF, BMP, DOCX, XLSX, and PPTX. The operation is asynchronous — it returns a job_id that you must poll using unsiloed_get_job_result to retrieve the parsed content.",
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "Publicly accessible URL to the document to parse."
                },
                "use_high_resolution": {
                    "type": "boolean",
                    "description": "Use high-resolution image processing for better OCR accuracy. Defaults to false."
                },
                "segmentation_method": {
                    "type": "string",
                    "enum": ["smart_layout_detection", "page_by_page"],
                    "description": "Document segmentation strategy. 'smart_layout_detection' (default) groups related elements into semantic chunks. 'page_by_page' creates one chunk per page."
                },
                "ocr_mode": {
                    "type": "string",
                    "enum": ["auto_ocr", "full_ocr"],
                    "description": "OCR strategy. 'auto_ocr' (default) only runs OCR when needed. 'full_ocr' forces OCR on all pages."
                },
                "merge_tables": {
                    "type": "boolean",
                    "description": "Merge adjacent table segments into a single table. Defaults to false."
                },
                "page_range": {
                    "type": "string",
                    "description": "Specify which pages to process. Formats: '1-5', '2,4,6', or '[1,3,5]'. Defaults to all pages."
                },
                "export_format": {
                    "type": "string",
                    "description": "JSON array of export format(s) to generate after parsing. Currently supported: ['docx']. Example: [\"docx\"]"
                }
            },
            "required": ["url"]
        }
    },
    {
        "name": "unsiloed_extract_data",
        "description": "Extract structured data from a PDF document using a custom JSON schema. Use this tool when the user wants to pull specific fields (such as invoice numbers, names, dates, or line items) out of a PDF. You must provide a publicly accessible URL to the PDF and a JSON schema defining the fields to extract. The operation is asynchronous — it returns a job_id that you must poll using unsiloed_get_job_result to retrieve the extracted data. Do not use this tool for document classification or splitting.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_url": {
                    "type": "string",
                    "description": "Publicly accessible URL to the PDF file to process."
                },
                "schema_data": {
                    "type": "string",
                    "description": "A JSON-stringified schema defining the fields to extract. Example: {\"type\":\"object\",\"properties\":{\"invoice_number\":{\"type\":\"string\",\"description\":\"The invoice number\"}},\"required\":[\"invoice_number\"],\"additionalProperties\":false}"
                },
                "model": {
                    "type": "string",
                    "enum": ["alpha", "beta", "gamma", "delta"],
                    "description": "Model tier for extraction. Default is 'gamma', recommended for most use cases."
                },
                "enable_citations": {
                    "type": "boolean",
                    "description": "When true, returns bounding box coordinates for each extracted value. Defaults to false."
                }
            },
            "required": ["file_url", "schema_data"]
        }
    },
    {
        "name": "unsiloed_classify_document",
        "description": "Classify a PDF document into one of several predefined categories. Use this tool when the user wants to determine what type of document a PDF is (for example, invoice, receipt, contract, or medical record). You must provide a publicly accessible URL to the PDF and a list of candidate categories. The operation is asynchronous — it returns a job_id that you must poll using unsiloed_get_job_result to retrieve the classification result with confidence scores. Do not use this for data extraction or document splitting.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_url": {
                    "type": "string",
                    "description": "Publicly accessible URL to the PDF file to classify."
                },
                "categories": {
                    "type": "string",
                    "description": "A JSON-stringified array of category objects. Each object must have a 'name' field and may have an optional 'description' field. Example: [{\"name\":\"Invoice\",\"description\":\"Financial invoices\"},{\"name\":\"Receipt\"}]"
                }
            },
            "required": ["file_url", "categories"]
        }
    },
    {
        "name": "unsiloed_split_document",
        "description": "Split a multi-document PDF into separate files by classifying each page into predefined categories. Use this tool when the user has a single PDF containing multiple document types and wants them separated into individual files. You must provide a publicly accessible URL to the PDF and a list of candidate categories. The operation is asynchronous — it returns a job_id that you must poll using unsiloed_get_job_result to retrieve download links for the split files. Do not use this for single-document classification or data extraction.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_url": {
                    "type": "string",
                    "description": "Publicly accessible URL to the PDF file to split."
                },
                "categories": {
                    "type": "string",
                    "description": "A JSON-stringified array of category objects. Each object must have a 'name' field and may have an optional 'description' field. Example: [{\"name\":\"Invoice\"},{\"name\":\"Contract\"},{\"name\":\"Receipt\"}]"
                }
            },
            "required": ["file_url", "categories"]
        }
    },
    {
        "name": "unsiloed_get_job_result",
        "description": "Poll for the result of an asynchronous Unsiloed job. Use this tool after calling unsiloed_parse_document, unsiloed_extract_data, unsiloed_classify_document, or unsiloed_split_document to check whether the job has completed and retrieve its results. If the status indicates the job is still processing, wait a few seconds and call this tool again. Once the job is complete, the response contains the output data. If the job failed, the response includes an error message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "job_id": {
                    "type": "string",
                    "description": "The job_id returned by a previous Unsiloed tool call."
                },
                "job_type": {
                    "type": "string",
                    "enum": ["parse", "extract", "classify", "splitter"],
                    "description": "The type of job to check. Use 'parse' for parsing jobs, 'extract' for extraction jobs, 'classify' for classification jobs, and 'splitter' for splitting jobs."
                }
            },
            "required": ["job_id", "job_type"]
        }
    }
]

print(f"Defined {len(tools)} tools: {[t['name'] for t in tools]}")

## 3. Build the Tool Executor

This function takes a tool name and input from Claude's response and makes the corresponding HTTP request to the Unsiloed API.

In [ ]:
def process_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute an Unsiloed API tool call and return the result as a JSON string."""

    if tool_name == "unsiloed_parse_document":
        data = {"url": tool_input["url"]}
        for key in ["use_high_resolution", "merge_tables"]:
            if key in tool_input:
                data[key] = str(tool_input[key]).lower()
        for key in ["segmentation_method", "ocr_mode", "page_range", "export_format"]:
            if key in tool_input:
                data[key] = tool_input[key]
        resp = requests.post(f"{UNSILOED_BASE_URL}/parse", headers=UNSILOED_HEADERS, data=data)

    elif tool_name == "unsiloed_extract_data":
        data = {
            "file_url": tool_input["file_url"],
            "schema_data": tool_input["schema_data"],
        }
        if "model" in tool_input:
            data["model"] = tool_input["model"]
        if "enable_citations" in tool_input:
            data["enable_citations"] = str(tool_input["enable_citations"]).lower()
        resp = requests.post(f"{UNSILOED_BASE_URL}/v2/extract", headers=UNSILOED_HEADERS, data=data)

    elif tool_name == "unsiloed_classify_document":
        data = {
            "file_url": tool_input["file_url"],
            "categories": tool_input["categories"],
        }
        resp = requests.post(f"{UNSILOED_BASE_URL}/classify", headers=UNSILOED_HEADERS, data=data)

    elif tool_name == "unsiloed_split_document":
        data = {
            "file_url": tool_input["file_url"],
            "categories": tool_input["categories"],
        }
        resp = requests.post(f"{UNSILOED_BASE_URL}/splitter", headers=UNSILOED_HEADERS, data=data)

    elif tool_name == "unsiloed_get_job_result":
        job_type = tool_input["job_type"]
        job_id = tool_input["job_id"]
        time.sleep(5)  # Brief pause before polling
        resp = requests.get(f"{UNSILOED_BASE_URL}/{job_type}/{job_id}", headers=UNSILOED_HEADERS)

    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})

    return json.dumps(resp.json())


print("Tool executor ready.")

## 4. Build the Agentic Loop

This is the core pattern: we send a message to Claude with our tools, and if Claude responds with `tool_use`, we execute those tool calls and feed the results back. This continues until Claude provides a final text response.

```
User message → Claude → tool_use? → execute tool → feed result back → Claude → ... → final text
```

In [ ]:
def run_agent(user_message: str, model: str = "claude-sonnet-4-20250514") -> str:
    """Run an agentic loop where Claude processes documents via Unsiloed tools."""
    messages = [{"role": "user", "content": user_message}]

    print(f"User: {user_message}\n")

    while True:
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            tools=tools,
            messages=messages,
        )

        # Add Claude's response to message history
        messages.append({"role": "assistant", "content": response.content})

        # If Claude is done, return the final text
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            print(f"Claude: {final_text}")
            return final_text

        # If Claude wants to use tools, execute them
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  → Calling {block.name}(...)")
                    result = process_tool_call(block.name, block.input)
                    # Truncate long results for display
                    display_result = result[:200] + "..." if len(result) > 200 else result
                    print(f"  ← Result: {display_result}\n")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})


print("Agentic loop ready.")

## 5. Example: Extract Invoice Data

Let's ask Claude to extract structured data from a PDF. Claude will:
1. Call `unsiloed_extract_data` with a schema defining the fields to extract
2. Call `unsiloed_get_job_result` to poll for the extraction results
3. Summarize the extracted data in natural language

> **Note:** Replace the URL below with a publicly accessible PDF URL to test with your own documents.

In [ ]:
# Replace with a real, publicly accessible PDF URL to test
PDF_URL = "https://example.com/sample-invoice.pdf"

result = run_agent(
    f"Extract the invoice number, date, vendor name, and total amount from this PDF: {PDF_URL}"
)

## 6. Example: Classify a Document

Now let's ask Claude to classify a document. Claude will:
1. Call `unsiloed_classify_document` with candidate categories
2. Poll for results
3. Report the classification with confidence scores

In [ ]:
# Replace with a real, publicly accessible PDF URL to test
PDF_URL = "https://example.com/sample-document.pdf"

result = run_agent(
    f"Classify this document into one of these categories: Invoice, Contract, Receipt, Report. "
    f"Here's the PDF: {PDF_URL}"
)

## 7. Example: Parse and Summarize

Finally, let's ask Claude to parse a document into structured chunks and then summarize the content. Claude will:
1. Call `unsiloed_parse_document` to break the document into segments
2. Poll for the parsed results
3. Read through the chunks and provide a summary

In [ ]:
# Replace with a real, publicly accessible document URL to test
DOC_URL = "https://example.com/sample-report.pdf"

result = run_agent(
    f"Parse this document and give me a summary of its contents, "
    f"including any tables or key data you find: {DOC_URL}"
)

## Next Steps

- **Full tool schemas with all parameters:** See the [Claude Integration docs page](https://docs.unsiloed.ai/document-processing/claude-integration) for the complete tool definitions including all optional parameters
- **API Reference:** Explore the full [Unsiloed API Reference](https://docs.unsiloed.ai/api-reference/extraction/extract-data) for detailed endpoint documentation
- **Anthropic Tool Use:** Read the [Anthropic tool-use guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview) for advanced patterns like tool choice, streaming, and error handling